<a href="https://colab.research.google.com/github/S-nakib30/Machine-Learning-and-AI-projects-/blob/main/Cross_OS_fedarel_unet_xgb_IDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross-OS Federated IDS: U-Net + XGBoost (CIC IoMT + ROS/MQTT)
**Blocks**
1. Setup & config  2. Load data  3. Prepare clients  4. Global labels  5. RF feature selection
6. Scale features  7. Model definition  8. Training helpers  9. Evaluation helpers
10. Local training - IoMT  11. Local training - ROS  12. Federated training  13. Global IDS (XGBoost)
14. Evaluate global IDS  15. Summary & save

Everything is saved under `/content/drive/MyDrive/cross_OS_analysis`.

In [ ]:
# 1. Setup: imports, mount Drive, output folders
import os, re, gc, json, copy, warnings
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import xgboost as xgb
from google.colab import drive
warnings.filterwarnings("ignore")
drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/cross_OS_analysis"
DIRS = {k: f"{OUT_DIR}/{k}" for k in ["features", "local_models", "federated", "global_ids", "reports"]}
for d in [OUT_DIR, *DIRS.values()]: os.makedirs(d, exist_ok=True)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| outputs ->", OUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda | outputs -> /content/drive/MyDrive/cross_OS_analysis


In [ ]:
# 2. Config: paths & hyper-parameters
IOMT_TRAIN = "/content/drive/MyDrive/CIC IoM Cyber Security Dataset/train_iomt.csv"
IOMT_TEST  = "/content/drive/MyDrive/CIC IoM Cyber Security Dataset/test_iomt.csv"
ROS_DIR    = "/content/drive/MyDrive/ROS_processed_FINAL_TRAIN/"
ROS_TRAIN_FILES = {                         # file -> class label
    f"/content/drive/MyDrive/ROS_processed_FINAL_TRAIN/normal.csv":                 "normal",
    f"{ROS_DIR}/DoS.csv":                    "DoS",
    f"{ROS_DIR}/subscriberflood.csv":        "subscriberflood",
    f"{ROS_DIR}/unauthorizedpublisher.csv":  "unauthorizedpublisher",
    f"{ROS_DIR}/unauthorizedsubscriber.csv": "unauthorizedsubscriber",
}
ROS_TEST ="/content/drive/MyDrive/ROS_Final_Combined_test/ROS_Combined_Test_v1.csv"

LABEL_CANDIDATES = ["label", "Label", "attack", "Attack", "attack_type", "class", "Class", "target"]
DROP_COLS        = ["Unnamed: 0", "timestamp", "Timestamp", "time", "Time", "id", "ID"]
COARSE_LABELS    = True      # merge similar attack names across datasets
MAX_PER_CLASS    = 60000     # cap train rows per class
RF_SAMPLE        = 100000
RF_TREES         = 100
K_FEATURES       = 32        # must be divisible by 4
LOCAL_EPOCHS     = 10
ROUNDS           = 10
FED_EPOCHS       = 2
FEDPROX_MU       = 0.01      # 0 -> plain FedAvg
AGG_WEIGHTING    = "samples" # "samples" | "equal"
GLOBAL_XGB_MODE  = "ensemble"# "ensemble" | "pooled"
XGB_PARAMS = dict(n_estimators=300, max_depth=8, learning_rate=0.1, subsample=0.8,
                  colsample_bytree=0.8, tree_method="hist", device=DEVICE, random_state=SEED)

json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, dict, list))},
          open(f"{OUT_DIR}/config.json", "w"), indent=2, default=str)


In [ ]:
# 3a. Label helpers
def unify_label(s):
    s = re.sub(r"[^a-z0-9]+", "_", str(s).strip().lower()).strip("_")
    if not COARSE_LABELS: return s
    if "benign" in s or "normal" in s: return "benign"
    if "unauthorized" in s and "publisher" in s:  return "unauthorized_publisher"
    if "unauthorized" in s and "subscriber" in s: return "unauthorized_subscriber"
    if "ddos" in s or s == "dos" or s.startswith("dos_") or "_dos" in s: return "dos"
    if "flood" in s: return "flood"
    if "scan" in s or "recon" in s: return "recon"
    if "spoof" in s: return "spoofing"
    if "malformed" in s: return "malformed"
    return s

def find_label_col(df):
    for c in LABEL_CANDIDATES:
        if c in df.columns: return c
    raise ValueError(f"No label column. Columns: {list(df.columns)[:40]} ... edit LABEL_CANDIDATES.")


In [ ]:
# 3b. Load IoMT
tr = pd.read_csv(IOMT_TRAIN, low_memory=False); te = pd.read_csv(IOMT_TEST, low_memory=False)
l_tr, l_te = find_label_col(tr), find_label_col(te)
iomt_raw = (tr, te, tr[l_tr], te[l_te], {l_tr, l_te}); del tr, te
print("IoMT label column:", l_tr, "| raw labels:", iomt_raw[2].nunique())


IoMT label column: label | raw labels: 18


In [ ]:
# 3c. Load ROS (train files + combined test)
parts = []
for path, lab in ROS_TRAIN_FILES.items():
    d = pd.read_csv(path, low_memory=False); d["__y__"] = lab; parts.append(d); print(f"{lab:24s} {len(d):>9,}")
ros_tr = pd.concat(parts, ignore_index=True); del parts
ros_te = pd.read_csv(ROS_TEST, low_memory=False); l_ros = find_label_col(ros_te)
drop = {"__y__", l_ros} | {c for c in LABEL_CANDIDATES if c in ros_tr.columns}
ros_raw = (ros_tr, ros_te, ros_tr["__y__"], ros_te[l_ros], drop); del ros_tr, ros_te
print("ROS test label column:", l_ros)


normal                      21,416
DoS                        600,015
subscriberflood             46,599
unauthorizedpublisher       14,791
unauthorizedsubscriber      12,991
ROS test label column: Label


In [ ]:
# 4. Prepare clients (clean, numeric-only, impute, cap per class)
def cap_per_class(y, cap, rng):
    idx = [rng.choice(np.where(y == c)[0], min((y == c).sum(), cap), replace=False) for c in np.unique(y)]
    return np.sort(np.concatenate(idx))

def prepare_client(name, tr, te, ytr_raw, yte_raw, drop):
    rng = np.random.default_rng(SEED)
    ytr, yte = ytr_raw.map(unify_label).values, yte_raw.map(unify_label).values
    cols = [c for c in tr.select_dtypes(include=[np.number]).columns if c not in drop and c not in DROP_COLS]
    Xtr = tr[cols].replace([np.inf, -np.inf], np.nan).astype("float32")
    Xte = te.reindex(columns=cols).apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float32")
    med = Xtr.median(); Xtr, Xte = Xtr.fillna(med).fillna(0), Xte.fillna(med).fillna(0)
    keep = Xtr.columns[Xtr.std() > 0]; Xtr, Xte = Xtr[keep], Xte[keep]
    i = cap_per_class(ytr, MAX_PER_CLASS, rng); Xtr, ytr = Xtr.iloc[i].reset_index(drop=True), ytr[i]
    print(f"[{name}] train {Xtr.shape} test {Xte.shape}\n  train: {dict(zip(*np.unique(ytr, return_counts=True)))}"
          f"\n  test : {dict(zip(*np.unique(yte, return_counts=True)))}")
    return dict(name=name, Xtr=Xtr, ytr_s=ytr, Xte=Xte, yte_s=yte)

iomt = prepare_client("IoMT", *iomt_raw); del iomt_raw
ros  = prepare_client("ROS",  *ros_raw);  del ros_raw
clients = [iomt, ros]; gc.collect()


[IoMT] train (201917, 44) test (1614182, 44)
  train: {'benign': np.int64(60000), 'dos': np.int64(60000), 'malformed': np.int64(5130), 'ping_sweep': np.int64(740), 'recon': np.int64(60000), 'spoofing': np.int64(16047)}
  test : {'benign': np.int64(37607), 'dos': np.int64(1545408), 'malformed': np.int64(1747), 'ping_sweep': np.int64(186), 'recon': np.int64(27490), 'spoofing': np.int64(1744)}
[ROS] train (155797, 65) test (69537, 65)
  train: {'benign': np.int64(21416), 'dos': np.int64(60000), 'flood': np.int64(46599), 'unauthorized_publisher': np.int64(14791), 'unauthorized_subscriber': np.int64(12991)}
  test : {'flood': np.int64(20000), 'noattack': np.int64(20000), 'unauthorized_publisher': np.int64(15302), 'unauthorized_subscriber': np.int64(14235)}


4

In [ ]:
# 5. Global label space (union across both datasets)
all_labels = sorted(set(iomt["ytr_s"]) | set(ros["ytr_s"]) | set(iomt["yte_s"]) | set(ros["yte_s"]))
le = LabelEncoder().fit(all_labels); N_CLASSES = len(le.classes_)
print("Global classes:", list(le.classes_))
for c in clients:
    c["ytr"] = le.transform(c["ytr_s"])
    known = np.isin(c["yte_s"], le.classes_)
    c["Xte"], c["yte_s"] = c["Xte"][known], c["yte_s"][known]; c["yte"] = le.transform(c["yte_s"])
joblib.dump(le, f"{DIRS['features']}/label_encoder.joblib")


Global classes: [np.str_('benign'), np.str_('dos'), np.str_('flood'), np.str_('malformed'), np.str_('noattack'), np.str_('ping_sweep'), np.str_('recon'), np.str_('spoofing'), np.str_('unauthorized_publisher'), np.str_('unauthorized_subscriber')]


['/content/drive/MyDrive/cross_OS_analysis/features/label_encoder.joblib']

In [ ]:
# 6. Random-Forest feature selection (per dataset)
def rf_select(c, k=K_FEATURES):
    rng = np.random.default_rng(SEED); n = min(RF_SAMPLE, len(c["Xtr"])); i = rng.choice(len(c["Xtr"]), n, replace=False)
    rf = RandomForestClassifier(n_estimators=RF_TREES, max_depth=20, n_jobs=-1,
                                class_weight="balanced_subsample", random_state=SEED).fit(c["Xtr"].iloc[i], c["ytr"][i])
    imp = pd.Series(rf.feature_importances_, index=c["Xtr"].columns).sort_values(ascending=False)
    return list(imp.index[:k]), imp, rf

for c in clients:
    c["feats"], c["imp"], rf = rf_select(c)
    print(f"[{c['name']}] top-10 RF features:\n{c['imp'].head(10).round(4).to_string()}\n")
    c["imp"].to_csv(f"{DIRS['features']}/{c['name']}_rf_importance.csv")
    json.dump(c["feats"], open(f"{DIRS['features']}/{c['name']}_selected_features.json", "w"), indent=1)
    joblib.dump(rf, f"{DIRS['features']}/{c['name']}_rf_selector.joblib"); del rf


[IoMT] top-10 RF features:
IAT              0.1005
rst_count        0.0925
Tot size         0.0648
Magnitue         0.0634
Header_Length    0.0582
Srate            0.0580
Rate             0.0512
fin_count        0.0499
AVG              0.0483
Min              0.0366

[ROS] top-10 RF features:
Dst Port             0.0803
Bwd Header Len       0.0658
Init Bwd Win Byts    0.0623
Bwd IAT Mean         0.0516
Fwd IAT Min          0.0492
Bwd IAT Max          0.0459
Bwd IAT Tot          0.0444
Tot Bwd Pkts         0.0398
Src Port             0.0367
Bwd Pkts/s           0.0317



In [ ]:
# 7. Scale selected features (+ pad to K) and save scalers
for c in clients:
    sc = StandardScaler().fit(c["Xtr"][c["feats"]])
    def tf(X, sc=sc, c=c):
        A = np.clip(sc.transform(X[c["feats"]]), -10, 10).astype("float32")
        return np.pad(A, ((0, 0), (0, K_FEATURES - A.shape[1]))) if A.shape[1] < K_FEATURES else A
    c["Xtr_s"], c["Xte_s"] = tf(c["Xtr"]), tf(c["Xte"])
    joblib.dump(sc, f"{DIRS['features']}/{c['name']}_scaler.joblib")
    print(c["name"], c["Xtr_s"].shape, c["Xte_s"].shape)


IoMT (201917, 32) (1614182, 32)
ROS (155797, 32) (69537, 32)


In [ ]:
# 8. Model: local adapter + shared 1D U-Net + shared head
def block(i, o):
    return nn.Sequential(nn.Conv1d(i, o, 3, padding=1), nn.BatchNorm1d(o), nn.ReLU(inplace=True),
                         nn.Conv1d(o, o, 3, padding=1), nn.BatchNorm1d(o), nn.ReLU(inplace=True))

class UNet1D(nn.Module):
    def __init__(self, b=16):
        super().__init__()
        self.enc1, self.enc2, self.bott = block(1, b), block(b, 2*b), block(2*b, 4*b)
        self.up2, self.dec2 = nn.ConvTranspose1d(4*b, 2*b, 2, stride=2), block(4*b, 2*b)
        self.up1, self.dec1 = nn.ConvTranspose1d(2*b, b, 2, stride=2),   block(2*b, b)
        self.emb_dim = 7 * b
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(F.max_pool1d(e1, 2)); bt = self.bott(F.max_pool1d(e2, 2))
        d2 = self.dec2(torch.cat([self.up2(bt), e2], 1)); d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return torch.cat([bt.mean(-1), d2.mean(-1), d1.mean(-1)], 1)

class ClientNet(nn.Module):
    def __init__(self, k=K_FEATURES, n_classes=N_CLASSES):
        super().__init__()
        self.adapter  = nn.Sequential(nn.Linear(k, k), nn.LayerNorm(k))           # LOCAL
        self.backbone = UNet1D()                                                  # SHARED
        self.head     = nn.Sequential(nn.Linear(self.backbone.emb_dim, 64), nn.ReLU(),
                                      nn.Dropout(0.2), nn.Linear(64, n_classes))  # SHARED
    def forward(self, x):
        emb = self.backbone(self.adapter(x).unsqueeze(1)); return self.head(emb), emb

def shared_state(m):
    return {k: v.detach().cpu().clone() for k, v in m.state_dict().items() if k.startswith(("backbone.", "head."))}


In [ ]:
# 9. Training helpers
def class_weights(y):
    cnt = np.bincount(y, minlength=N_CLASSES).astype("float64"); w = np.ones(N_CLASSES); p = cnt > 0
    w[p] = np.sqrt(cnt[p].sum() / (p.sum() * cnt[p])); return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def train_local(model, X, y, epochs, lr=1e-3, bs=512, ref=None, mu=0.0):
    model.to(DEVICE).train(); opt = torch.optim.Adam(model.parameters(), lr=lr); cw = class_weights(y)
    dl = DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(y).long()), batch_size=bs, shuffle=True, drop_last=len(y) > bs)
    if ref is not None: ref = {k: v.to(DEVICE) for k, v in ref.items()}
    for _ in range(epochs):
        tot = 0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); logits, _ = model(xb); loss = F.cross_entropy(logits, yb, weight=cw)
            if ref is not None and mu > 0:
                loss = loss + (mu / 2) * sum(((p - ref[n]) ** 2).sum() for n, p in model.named_parameters() if n in ref)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(yb)
    return tot / len(y)

@torch.no_grad()
def infer(model, X, bs=4096):
    model.to(DEVICE).eval(); L, E = [], []
    for i in range(0, len(X), bs):
        lg, em = model(torch.from_numpy(X[i:i+bs]).to(DEVICE)); L.append(lg.cpu()); E.append(em.cpu())
    return torch.cat(L).numpy(), torch.cat(E).numpy()

def fit_xgb(X, y):
    uniq = np.unique(y); remap = {c: i for i, c in enumerate(uniq)}
    clf = xgb.XGBClassifier(objective="multi:softprob" if len(uniq) > 2 else "binary:logistic", **XGB_PARAMS)
    clf.fit(X, np.vectorize(remap.get)(y)); return clf, uniq

def proba_global(clf, uniq, X):
    p = clf.predict_proba(X); out = np.zeros((len(X), N_CLASSES), dtype="float32")
    out[:, uniq] = p; return out


In [ ]:
# 10. Evaluation helpers (print + save report CSV + confusion-matrix PNG)
results = []
def report(title, y_true, y_pred, tag):
    labs = np.unique(np.concatenate([y_true, y_pred])); names = le.classes_[labs]
    acc = accuracy_score(y_true, y_pred); mf1 = f1_score(y_true, y_pred, average="macro"); wf1 = f1_score(y_true, y_pred, average="weighted")
    print(f"\n=== {title} ===  acc={acc:.4f} macroF1={mf1:.4f} weightedF1={wf1:.4f}")
    print(classification_report(y_true, y_pred, labels=labs, target_names=names, digits=4, zero_division=0))
    rep = classification_report(y_true, y_pred, labels=labs, target_names=names, digits=4, zero_division=0, output_dict=True)
    pd.DataFrame(rep).T.to_csv(f"{DIRS['reports']}/{tag}_classification_report.csv")
    cm = confusion_matrix(y_true, y_pred, labels=labs); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(max(5, .6*len(labs)+3),) * 2); ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labs))); ax.set_yticks(range(len(labs))); ax.set_xticklabels(names, rotation=60, ha="right"); ax.set_yticklabels(names)
    for i in range(len(labs)):
        for j in range(len(labs)): ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cmn[i, j] > .5 else "black")
    ax.set_title(title, fontsize=9); ax.set_xlabel("Predicted"); ax.set_ylabel("True"); fig.tight_layout()
    fig.savefig(f"{DIRS['reports']}/{tag}_confusion.png", dpi=130); plt.close(fig)
    results.append(dict(model=title, acc=acc, macro_f1=mf1, weighted_f1=wf1))


In [ ]:
# 11. Local training - IoMT (U-Net -> XGBoost on [selected features | embedding])
def train_local_pipeline(c):
    print(f"\n######## Local training: {c['name']} ########")
    m = ClientNet()
    for ep in range(0, LOCAL_EPOCHS, 2):
        print(f"  epochs {ep+2}/{LOCAL_EPOCHS} loss={train_local(m, c['Xtr_s'], c['ytr'], 2):.4f}")
    _, Etr = infer(m, c["Xtr_s"]); lg, Ete = infer(m, c["Xte_s"])
    report(f"{c['name']} local U-Net only", c["yte"], lg.argmax(1), f"{c['name']}_local_unet")
    clf, uniq = fit_xgb(np.hstack([c["Xtr_s"], Etr]), c["ytr"])
    report(f"{c['name']} local U-Net+XGBoost", c["yte"], proba_global(clf, uniq, np.hstack([c["Xte_s"], Ete])).argmax(1), f"{c['name']}_local_unet_xgb")
    torch.save(m.state_dict(), f"{DIRS['local_models']}/{c['name']}_unet.pt"); clf.save_model(f"{DIRS['local_models']}/{c['name']}_xgb.json")
    np.save(f"{DIRS['local_models']}/{c['name']}_xgb_classes.npy", uniq)

train_local_pipeline(iomt)



######## Local training: IoMT ########
  epochs 2/10 loss=0.2978
  epochs 4/10 loss=0.2743
  epochs 6/10 loss=0.2636
  epochs 8/10 loss=0.2564
  epochs 10/10 loss=0.2505

=== IoMT local U-Net only ===  acc=0.9941 macroF1=0.6914 weightedF1=0.9953
              precision    recall  f1-score   support

      benign     0.9557    0.8523    0.9011     37607
         dos     1.0000    0.9991    0.9995   1545408
   malformed     0.6014    0.8266    0.6962      1747
  ping_sweep     0.1761    0.8548    0.2920       186
       recon     0.9770    0.9367    0.9564     27490
    spoofing     0.1895    0.7603    0.3033      1744

    accuracy                         0.9941   1614182
   macro avg     0.6499    0.8716    0.6914   1614182
weighted avg     0.9972    0.9941    0.9953   1614182


=== IoMT local U-Net+XGBoost ===  acc=0.9979 macroF1=0.8633 weightedF1=0.9981
              precision    recall  f1-score   support

      benign     0.9776    0.9588    0.9681     37607
         dos     1.000

In [ ]:
# 12. Local training - ROS
train_local_pipeline(ros)



######## Local training: ROS ########
  epochs 2/10 loss=0.5978
  epochs 4/10 loss=0.5045
  epochs 6/10 loss=0.4847
  epochs 8/10 loss=0.4743
  epochs 10/10 loss=0.4646

=== ROS local U-Net only ===  acc=0.0346 macroF1=0.0420 weightedF1=0.0562
                         precision    recall  f1-score   support

                 benign     0.0000    0.0000    0.0000         0
                    dos     0.0000    0.0000    0.0000         0
                  flood     0.3877    0.0177    0.0339     20000
               noattack     0.0000    0.0000    0.0000     20000
 unauthorized_publisher     0.2005    0.0803    0.1147     15302
unauthorized_subscriber     0.4882    0.0580    0.1036     14235

               accuracy                         0.0346     69537
              macro avg     0.1794    0.0260    0.0420     69537
           weighted avg     0.2556    0.0346    0.0562     69537


=== ROS local U-Net+XGBoost ===  acc=0.0208 macroF1=0.0244 weightedF1=0.0339
                        

In [ ]:
# 13a. Federated setup (FedAvg helper, fresh client models, common init)
def fedavg(states, weights):
    w = np.array(weights, dtype="float64"); w /= w.sum()
    return {k: (sum(wi * s[k].float() for wi, s in zip(w, states)) if states[0][k].is_floating_point() else states[0][k].clone()) for k in states[0]}

for c in clients: c["model"] = ClientNet().to(DEVICE)
global_shared = fedavg([shared_state(c["model"]) for c in clients], [1, 1])
agg_w = [len(c["ytr"]) if AGG_WEIGHTING == "samples" else 1 for c in clients]
fed_log = []


In [ ]:
# 13b. Federated rounds (checkpoint saved every round)
for r in range(1, ROUNDS + 1):
    states = []
    for c in clients:
        c["model"].load_state_dict(global_shared, strict=False)      # adapters stay local
        c["loss"] = train_local(c["model"], c["Xtr_s"], c["ytr"], FED_EPOCHS, ref=global_shared, mu=FEDPROX_MU)
        states.append(shared_state(c["model"]))
    global_shared = fedavg(states, agg_w)
    row = {"round": r}
    for c in clients:
        c["model"].load_state_dict(global_shared, strict=False)
        lg, _ = infer(c["model"], c["Xte_s"])
        row[f"{c['name']}_loss"] = c["loss"]; row[f"{c['name']}_test_acc"] = accuracy_score(c["yte"], lg.argmax(1))
    fed_log.append(row); print(row)
    torch.save({"round": r, "global_shared": global_shared, **{f"{c['name']}_adapter": c["model"].adapter.state_dict() for c in clients}},
               f"{DIRS['federated']}/round_{r:02d}.pt")
torch.save(global_shared, f"{DIRS['federated']}/global_shared_final.pt")
for c in clients: torch.save(c["model"].adapter.state_dict(), f"{DIRS['federated']}/{c['name']}_adapter_final.pt")
fed_df = pd.DataFrame(fed_log); fed_df.to_csv(f"{DIRS['federated']}/federated_training_log.csv", index=False)


{'round': 1, 'IoMT_loss': 0.37723423412736384, 'IoMT_test_acc': 0.6363322103703299, 'ROS_loss': 0.6892598458952194, 'ROS_test_acc': 0.0}
{'round': 2, 'IoMT_loss': 0.30827067037996225, 'IoMT_test_acc': 0.9794744334901516, 'ROS_loss': 0.55793343591036, 'ROS_test_acc': 0.0}
{'round': 3, 'IoMT_loss': 0.29274984641323026, 'IoMT_test_acc': 0.9215528360494666, 'ROS_loss': 0.5343443427003882, 'ROS_test_acc': 0.0}
{'round': 4, 'IoMT_loss': 0.28226143182972074, 'IoMT_test_acc': 0.993200890605892, 'ROS_loss': 0.5180292107173994, 'ROS_test_acc': 0.0}
{'round': 5, 'IoMT_loss': 0.27891963256188457, 'IoMT_test_acc': 0.9922772029424192, 'ROS_loss': 0.5071271209568907, 'ROS_test_acc': 0.0}
{'round': 6, 'IoMT_loss': 0.27246636789443257, 'IoMT_test_acc': 0.9901498096249369, 'ROS_loss': 0.49523791152653523, 'ROS_test_acc': 0.0}
{'round': 7, 'IoMT_loss': 0.26717611016340503, 'IoMT_test_acc': 0.9915963627397654, 'ROS_loss': 0.48866930120994284, 'ROS_test_acc': 4.3142499676431255e-05}
{'round': 8, 'IoMT_loss

In [ ]:
# 13c. Plot federated convergence
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for c in clients:
    ax[0].plot(fed_df["round"], fed_df[f"{c['name']}_loss"], label=c["name"]); ax[1].plot(fed_df["round"], fed_df[f"{c['name']}_test_acc"], label=c["name"])
ax[0].set_title("Local loss per round"); ax[1].set_title("Global model test accuracy per round")
for a in ax: a.set_xlabel("Round"); a.legend()
fig.tight_layout(); fig.savefig(f"{DIRS['reports']}/federated_convergence.png", dpi=130); plt.show()


In [ ]:
# 14. Global IDS: global U-Net embeddings -> XGBoost
for c in clients:
    c["model"].load_state_dict(global_shared, strict=False)
    _, c["Etr"] = infer(c["model"], c["Xtr_s"]); c["lg_te"], c["Ete"] = infer(c["model"], c["Xte_s"])

if GLOBAL_XGB_MODE == "ensemble":        # each client trains its own XGB; server soft-votes
    for c in clients:
        c["xgb"], c["uniq"] = fit_xgb(c["Etr"], c["ytr"]); c["xgb"].save_model(f"{DIRS['global_ids']}/{c['name']}_xgb_on_global_emb.json")
        np.save(f"{DIRS['global_ids']}/{c['name']}_xgb_classes.npy", c["uniq"])
    wts = np.array([len(c["ytr"]) for c in clients], dtype="float64"); wts /= wts.sum()
    def global_ids_proba(E): return sum(w * proba_global(c["xgb"], c["uniq"], E) for w, c in zip(wts, clients))
else:                                    # one XGB on pooled embeddings
    g_xgb, g_uniq = fit_xgb(np.vstack([c["Etr"] for c in clients]), np.concatenate([c["ytr"] for c in clients]))
    g_xgb.save_model(f"{DIRS['global_ids']}/pooled_xgb.json"); np.save(f"{DIRS['global_ids']}/pooled_xgb_classes.npy", g_uniq)
    def global_ids_proba(E): return proba_global(g_xgb, g_uniq, E)


In [ ]:
# 15. Evaluate the global IDS on both test sets
for c in clients:
    report(f"GLOBAL U-Net head on {c['name']} test", c["yte"], c["lg_te"].argmax(1), f"global_unet_{c['name']}")
    report(f"GLOBAL IDS (U-Net+XGB {GLOBAL_XGB_MODE}) on {c['name']} test", c["yte"], global_ids_proba(c["Ete"]).argmax(1), f"global_ids_{c['name']}")



=== GLOBAL U-Net head on IoMT test ===  acc=0.9903 macroF1=0.4517 weightedF1=0.9913
                         precision    recall  f1-score   support

                 benign     0.7727    0.8942    0.8290     37607
                    dos     1.0000    0.9944    0.9972   1545408
                  flood     0.0000    0.0000    0.0000         0
              malformed     0.5837    0.6090    0.5961      1747
             ping_sweep     0.2168    0.7366    0.3350       186
                  recon     0.9737    0.9341    0.9535     27490
               spoofing     0.2393    0.6852    0.3548      1744
 unauthorized_publisher     0.0000    0.0000    0.0000         0
unauthorized_subscriber     0.0000    0.0000    0.0000         0

               accuracy                         0.9903   1614182
              macro avg     0.4207    0.5393    0.4517   1614182
           weighted avg     0.9929    0.9903    0.9913   1614182


=== GLOBAL IDS (U-Net+XGB ensemble) on IoMT test ===  acc=0.9943 m

In [ ]:
# 16. Summary & final save
summary = pd.DataFrame(results).round(4); print(summary.to_string(index=False))
summary.to_csv(f"{OUT_DIR}/results_summary.csv", index=False)
json.dump(list(le.classes_), open(f"{OUT_DIR}/global_classes.json", "w"))
print("\nSaved to", OUT_DIR); print(*sorted(os.listdir(OUT_DIR)), sep="\n  ")


                                       model    acc  macro_f1  weighted_f1
                       IoMT local U-Net only 0.9941    0.6914       0.9953
                    IoMT local U-Net+XGBoost 0.9979    0.8633       0.9981
                        ROS local U-Net only 0.0346    0.0420       0.0562
                     ROS local U-Net+XGBoost 0.0208    0.0244       0.0339
              GLOBAL U-Net head on IoMT test 0.9903    0.4517       0.9913
GLOBAL IDS (U-Net+XGB ensemble) on IoMT test 0.9943    0.5049       0.9955
               GLOBAL U-Net head on ROS test 0.0143    0.0115       0.0243
 GLOBAL IDS (U-Net+XGB ensemble) on ROS test 0.0071    0.0071       0.0135

Saved to /content/drive/MyDrive/cross_OS_analysis
config.json
  features
  federated
  global_classes.json
  global_ids
  local_models
  reports
  results_summary.csv
